# Preparation

In [ ]:
import logging
from pathlib import Path

import myo
import myoktros
import numpy as np
import seaborn as sn
from sklearn.metrics import accuracy_score, confusion_matrix

In [ ]:
# global variables
arm_dominance = "right"
assets_path = Path.cwd() / "assets"
config_path = Path.cwd() / "config.ini"
data_path = Path.cwd() / "data"
emg_mode = myo.types.EMGMode.SEND_FILT
n_samples = 25
k = 3
np.set_printoptions(precision=3, suppress=True)
myoktros.Gesture.load_config(config_path)

In [ ]:
test_data_path = Path.cwd() / "tests" / "data"

x_test = myoktros.GestureModel.read_data_agg(test_data_path, arm_dominance, emg_mode, n_samples)
y_test = x_test.pop('gesture')

# Train the models

In [ ]:
myoktros.KerasSequentialModel.fit(
    arm_dominance,
    assets_path,
    data_path,
    emg_mode,
    n_samples,
)
ksm = myoktros.KerasSequentialModel(
    arm_dominance,
    assets_path,
    emg_mode,
    n_samples,
)

In [ ]:
myoktros.KNNClassifier.fit(
    arm_dominance,
    assets_path,
    data_path,
    emg_mode,
    k,
    n_samples,    
)
kc = myoktros.KNNClassifier(
    arm_dominance,
    assets_path,
    emg_mode,
    n_samples,
)

# Visualize the classification performance vs. test data

In [ ]:
# keras
predictions = ksm.model.predict(x_test)
predicted_labels = np.argmax(predictions, axis=1)
cm = confusion_matrix(y_test, predicted_labels, normalize="pred")

# plot heatmap
ax = sn.heatmap(cm, annot=True, cmap="Blues", xticklabels=myoktros.Gesture.names, yticklabels=myoktros.Gesture.names)
_ = ax.set(xlabel="Predicted", ylabel="Actual")
# _ = ax.xaxis.tick_top()

In [ ]:
# knn
predicted_labels = kc.model.predict(x_test)
cm = confusion_matrix(y_test, predicted_labels, normalize="pred")

# plot heatmap
ax = sn.heatmap(cm, annot=True, cmap="Blues", xticklabels=myoktros.Gesture.names, yticklabels=myoktros.Gesture.names)
_ = ax.set(xlabel="Predicted", ylabel="Actual")
# _ = ax.xaxis.tick_top()

# Find out the optimal n_samples and K for k-NN

In [ ]:
results = []
for k in range(1, 10, 2):
    for n_samples in range(25, 201, 25):
        myoktros.KNNClassifier.fit(
            arm_dominance,
            assets_path,
            data_path,
            emg_mode,
            k,
            n_samples,    
        )
        kc = myoktros.KNNClassifier(
            arm_dominance,
            assets_path,
            emg_mode,
            n_samples,
        )
        predicted_labels = kc.model.predict(x_test)
        acc = accuracy_score(y_test, predicted_labels)
        results.append((k, n_samples, acc))
        # print(f"k: {k}, n_samples: {n_samples}: {acc:.3f}")

best = max(results, key=lambda r: r[2])
print(f"best combination – k: {best[0]}, n_samples: {best[1]} at {best[2]}")